# MAuto Multi-Role Diagnostics

Notebook này dùng admin API của `diagnostic_controller.py` để kiểm tra live `A/B/C/D` với Tampermonkey script hiện tại.

Các action hiện có trong script:
- `PROBE`
- `DUMP_BUTTONS`
- `WAIT_COMPOSER_STABLE`
- `SET_PROMPT`
- `FIND_SEND`
- `CLICK_SEND`
- `WAIT_ASSISTANT_DONE`
- `SYNC_TRANSCRIPT`

Mục tiêu:
- xác minh textarea detect đúng và ổn định
- xác minh nút send không biến mất khi cần bấm
- stress test với 4 role `A/B/C/D`
- có cả test trả lời rất ngắn và test trả lời dài hơn

In [ ]:
# BLOCK 1: imports + config
import json
import random
import time
import urllib.request
import urllib.parse
from pathlib import Path

BASE_URL = "http://127.0.0.1:8500"
ROLES = ["A", "B", "C", "D"]
RUN_DIR = Path(".")

print("BASE_URL:", BASE_URL)
print("ROLES:", ROLES)

In [ ]:
# BLOCK 2: low-level HTTP + helpers
def http_json(method, path, payload=None, timeout=300):
    url = f"{BASE_URL}{path}"
    data = None
    headers = {}
    if payload is not None:
        data = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        headers["Content-Type"] = "application/json"
    req = urllib.request.Request(url, data=data, headers=headers, method=method)
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))

def get_config():
    return http_json("GET", "/api/admin/config")["config"]

def update_config(updates):
    return http_json("POST", "/api/admin/config", {"config": updates})["config"]

def role_snapshot(role):
    return http_json("GET", f"/api/admin/role/{urllib.parse.quote(role)}")

def event_tail(role="", contains="", command_id="", limit=50):
    params = urllib.parse.urlencode({
        "role": role,
        "contains": contains,
        "command_id": command_id,
        "limit": limit,
    })
    return http_json("GET", f"/api/admin/events?{params}")

def send_command(role, action, payload=None):
    return http_json("POST", "/api/admin/command", {
        "role": role,
        "action": action,
        "payload": payload or {},
    })["command"]

def wait_command(command_id, timeout=300, print_every=1.0):
    started = time.time()
    last_print = 0
    while True:
        data = http_json("GET", f"/api/admin/command/{command_id}")
        now = time.time()
        if now - last_print >= print_every:
            print(f"[{time.strftime('%H:%M:%S')}] cmd={command_id[:8]} status={data['status']} done={data['done']}")
            last_print = now
        if data["done"]:
            return data["result"]
        if timeout and now - started > timeout:
            raise TimeoutError(f"Timeout waiting for command_id={command_id}")
        time.sleep(0.2)

def run_command(role, action, payload=None, timeout=300, print_every=1.0):
    cmd = send_command(role, action, payload)
    print(f"{action} -> role={role}, command_id={cmd['command_id']}")
    return wait_command(cmd["command_id"], timeout=timeout, print_every=print_every)

def dom_summary(dom):
    messages = dom.get("messages") or {}
    counts = messages.get("counts") if isinstance(messages, dict) else None
    return {
        "composer": dom.get("composer"),
        "composer_text_len": dom.get("composer_text_len"),
        "send_enabled": dom.get("send_enabled"),
        "stop_visible": dom.get("stop_visible"),
        "voice_visible": dom.get("voice_visible"),
        "selection_strategy": dom.get("selection_strategy"),
        "matched_selector": dom.get("matched_selector"),
        "composer_path": dom.get("composer_path"),
        "message_counts": counts,
    }

def print_config():
    print(json.dumps(get_config(), ensure_ascii=False, indent=2))

print("Helpers ready")

In [ ]:
# BLOCK 2.5: inspect or update live config from Python
print("Current config:")
print_config()

# Example:
# update_config({
#     "action_delay_min_ms": 3000,
#     "action_delay_max_ms": 5000,
#     "send_delay_min_ms": 2000,
#     "send_delay_max_ms": 5000,
#     "role_switch_delay_min_s": 3,
#     "role_switch_delay_max_s": 5,
# })

In [ ]:
# BLOCK 3: wait roles online + baseline snapshots
for role in ROLES:
    print("=" * 100)
    print("ROLE", role)
    print("=" * 100)
    for i in range(30):
        snap = role_snapshot(role)
        dom = snap.get("dom_info", {})
        print(f"[{i}] status={snap['status']} sessions={snap['sessions']}")
        if snap["status"] != "OFFLINE" or dom:
            print(json.dumps(dom_summary(dom), ensure_ascii=False, indent=2))
            break
        time.sleep(1)
    else:
        raise RuntimeError(f"Role {role} did not come online")

In [ ]:
# BLOCK 4: baseline PROBE + WAIT_COMPOSER_STABLE + DUMP_BUTTONS
baseline = {}
for role in ROLES:
    print("\n" + "=" * 100)
    print(f"BASELINE {role}")
    print("=" * 100)
    probe = run_command(role, "PROBE", timeout=60)
    stable = run_command(role, "WAIT_COMPOSER_STABLE", {"samples": 10, "sample_ms": 300}, timeout=60)
    dump = run_command(role, "DUMP_BUTTONS", timeout=60)
    baseline[role] = {"probe": probe, "stable": stable, "dump": dump}
    print("PROBE:", probe["state"])
    print(json.dumps(dom_summary(probe.get("dom_info", {})), ensure_ascii=False, indent=2))
    print("STABLE:", stable["state"])
    print(json.dumps(stable.get("result", {}), ensure_ascii=False, indent=2))
    print("DUMP top button summary:")
    print(json.dumps((dump.get("result", {}) or {}).get("selected_button"), ensure_ascii=False, indent=2))

In [ ]:
# BLOCK 5: turn runner
def random_role_switch_sleep():
    cfg = get_config()
    delay_s = random.randint(int(cfg.get("role_switch_delay_min_s", 3)), int(cfg.get("role_switch_delay_max_s", 5)))
    print(f"Role switch sleep: {delay_s}s")
    time.sleep(delay_s)
    return delay_s

def run_turn(role, prompt_text, timeout=240):
    ready = run_command(role, "WAIT_COMPOSER_STABLE", {"samples": 6, "sample_ms": 300}, timeout=60, print_every=3)
    set_prompt = run_command(role, "SET_PROMPT", {
        "text": prompt_text,
        "method": "auto",
        "samples": 6,
        "sample_ms": 300,
    }, timeout=120, print_every=3)
    find_send = run_command(role, "FIND_SEND", timeout=60, print_every=3)
    click_send = run_command(role, "CLICK_SEND", timeout=60, print_every=3)
    assistant = run_command(role, "WAIT_ASSISTANT_DONE", timeout=timeout, print_every=2)
    sync = run_command(role, "SYNC_TRANSCRIPT", {"reason": "post_turn_sync"}, timeout=60, print_every=3)
    return {
        "ready": ready,
        "set_prompt": set_prompt,
        "find_send": find_send,
        "click_send": click_send,
        "assistant": assistant,
        "sync": sync,
        "response_text": (assistant.get("text") or "").strip(),
    }

def turn_summary(item):
    return {
        "ready": item["ready"]["state"],
        "set_prompt": item["set_prompt"]["state"],
        "find_send": item["find_send"]["state"],
        "selection_strategy": (item["find_send"].get("result") or {}).get("selection_strategy"),
        "matched_selector": (item["find_send"].get("result") or {}).get("matched_selector"),
        "click_send": item["click_send"]["state"],
        "assistant": item["assistant"]["state"],
        "response_text": item["response_text"],
    }

print("Turn runner ready")

In [ ]:
# BLOCK 6: stress test 7 loops, short answers
counting_log = []
n = 1

for loop_no in range(1, 8):
    print("\n" + "#" * 100)
    print(f"SHORT LOOP {loop_no}")
    print("#" * 100)
    for role in ROLES:
        prompt_text = f"Tra loi dung 1 ky tu so, khong them gi khac: {n}"
        switch_delay_s = random_role_switch_sleep()
        print("\n" + "-" * 100)
        print(f"TURN role={role} n={n}")
        print("-" * 100)
        item = run_turn(role, prompt_text, timeout=240)
        counting_log.append({
            "loop": loop_no,
            "role": role,
            "n": n,
            "switch_delay_s": switch_delay_s,
            "prompt": prompt_text,
            "result": item,
        })
        print(json.dumps(turn_summary(item), ensure_ascii=False, indent=2))
        n += 1

counting_path = RUN_DIR / "counting_abcd_7loops_log.json"
counting_path.write_text(json.dumps(counting_log, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", counting_path.resolve())

In [ ]:
# BLOCK 7: stress test 7 loops, longer poem answers
poem_log = []
poem_themes = {
    "A": "buoi sang",
    "B": "con song",
    "C": "con pho",
    "D": "mua he",
}

for loop_no in range(1, 8):
    print("\n" + "#" * 100)
    print(f"POEM LOOP {loop_no}")
    print("#" * 100)
    for role in ROLES:
        theme = poem_themes[role]
        switch_delay_s = random_role_switch_sleep()
        prompt_text = (
            f"Hay viet dung 1 cau tho tieng Viet dai 12-20 tu ve chu de {theme}, "
            f"co nhac so {loop_no}, khong giai thich, khong xuong dong."
        )
        print("\n" + "-" * 100)
        print(f"TURN role={role} poem_loop={loop_no}")
        print("-" * 100)
        item = run_turn(role, prompt_text, timeout=240)
        poem_log.append({
            "loop": loop_no,
            "role": role,
            "switch_delay_s": switch_delay_s,
            "prompt": prompt_text,
            "result": item,
        })
        print(json.dumps(turn_summary(item), ensure_ascii=False, indent=2))

poem_path = RUN_DIR / "poem_abcd_7loops_log.json"
poem_path.write_text(json.dumps(poem_log, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", poem_path.resolve())

In [ ]:
# BLOCK 8: summarize logs quickly
def summarize_log(entries):
    rows = []
    for entry in entries:
        item = entry["result"]
        rows.append({
            "loop": entry["loop"],
            "role": entry["role"],
            "ready": item["ready"]["state"],
            "set_prompt": item["set_prompt"]["state"],
            "find_send": item["find_send"]["state"],
            "selection_strategy": (item["find_send"].get("result") or {}).get("selection_strategy"),
            "matched_selector": (item["find_send"].get("result") or {}).get("matched_selector"),
            "click_send": item["click_send"]["state"],
            "assistant": item["assistant"]["state"],
            "response_len": len(item["response_text"]),
            "response_preview": item["response_text"][:120],
        })
    return rows

print(json.dumps(summarize_log(counting_log), ensure_ascii=False, indent=2)[:12000])

In [ ]:
# BLOCK 9: event tail for stalled turns
events = event_tail(limit=80)
for event in events["events"]:
    print(json.dumps(event, ensure_ascii=False)[:4000])